In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [10]:
import pandas as pd
import numpy as np
import warnings; warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.pipeline      import Pipeline
from sklearn.compose       import ColumnTransformer
from sklearn.impute        import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble      import RandomForestClassifier
from sklearn.metrics       import (classification_report,
                                      confusion_matrix,
                                      accuracy_score)
import joblib

print("All libraries imported ✓")

All libraries imported ✓


In [3]:
df = pd.read_csv('/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print("\nChurn distribution:")
print(df['Churn'].value_counts())
print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
df.head()

Shape: (7043, 21)
Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Churn distribution:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Missing values:
Series([], dtype: int64)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [13]:
# Fix TotalCharges: stored as string, blank for new customers
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Average monthly spend over customer's lifetime
df['AvgMonthlySpend'] = df['TotalCharges'] / (df['tenure'] + 1)

# New customers (first 6 months = highest churn risk window)
df['IsNewCustomer'] = (df['tenure'] <= 6).astype(int)

# Count of active services per customer
service_cols = ['PhoneService', 'MultipleLines', 'InternetService',
                'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
df['NumServices'] = df[service_cols].apply(
    lambda row: sum(v not in ['No', 'No internet service', 'No phone service']
                    for v in row), axis=1)

print(f"Dataset shape after engineering: {df.shape}")
print("New features: AvgMonthlySpend, IsNewCustomer, NumServices")

Dataset shape after engineering: (7043, 25)
New features: AvgMonthlySpend, IsNewCustomer, NumServices


In [28]:
X = df.drop('Churn', axis=1)
y = df['Churn']   # 'Yes' / 'No' — RandomForest handles string labels fine

numeric_features = [
    'tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen',
    'AvgMonthlySpend', 'IsNewCustomer', 'NumServices', 'NoProtection'
]

categorical_features = [
    'gender', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod', 'ChargesTier'
]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {len(X_train)} records | Test: {len(X_test)} records")


Train: 5634 records | Test: 1409 records


In [29]:
#Preprocessing sub-pipelines
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline,    numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

#XGBoost pipeline with SMOTE for class imbalance
model_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote',        SMOTE(random_state=42, k_neighbors=5)),
    ('model',        xgb.XGBClassifier(
        n_estimators    = 500,
        max_depth       = 5,
        learning_rate   = 0.05,
        subsample       = 0.8,
        colsample_bytree= 0.8,
        use_label_encoder=False,
        eval_metric     = 'logloss',
        random_state    = 42
    ))
])

print("Pipeline assembled: Preprocessor → SMOTE → XGBoostClassifier")


Pipeline assembled: Preprocessor → SMOTE → XGBoostClassifier


In [30]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline,    numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

# Final pipeline: exactly as required by the assignment 
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators = 100,
        max_depth    = 10,
        min_samples_leaf = 4,
        class_weight = 'balanced',
        random_state = 42
    ))
])

print("Pipeline ready: Preprocessor → RandomForestClassifier")

Pipeline ready: Preprocessor → RandomForestClassifier


In [31]:
model_pipeline.fit(X_train, y_train)
print("Training complete ✓")

Training complete ✓


In [32]:
y_pred = model_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.7679205110007097

Confusion Matrix:
[[800 235]
 [ 92 282]]

Classification Report:
              precision    recall  f1-score   support

          No       0.90      0.77      0.83      1035
         Yes       0.55      0.75      0.63       374

    accuracy                           0.77      1409
   macro avg       0.72      0.76      0.73      1409
weighted avg       0.80      0.77      0.78      1409



In [20]:
joblib.dump(model_pipeline, 'customer_churn_pipeline.pkl')
print("Pipeline saved ✓")

loaded_pipeline = joblib.load('customer_churn_pipeline.pkl')

new_customer = pd.DataFrame({
    'gender': ['Female'], 'SeniorCitizen': [0],
    'Partner': ['No'], 'Dependents': ['No'],
    'tenure': [3], 'PhoneService': ['Yes'],
    'MultipleLines': ['No'], 'InternetService': ['Fiber optic'],
    'OnlineSecurity': ['No'], 'OnlineBackup': ['No'],
    'DeviceProtection': ['No'], 'TechSupport': ['No'],
    'StreamingTV': ['Yes'], 'StreamingMovies': ['Yes'],
    'Contract': ['Month-to-month'],
    'PaperlessBilling': ['Yes'],
    'PaymentMethod': ['Electronic check'],
    'MonthlyCharges': [95.75], 'TotalCharges': [287.25],
    'AvgMonthlySpend': [71.8], 'IsNewCustomer': [1],
    'NumServices': [4]
})

prediction = loaded_pipeline.predict(new_customer)
print(f"\nNew Customer Churn Prediction: {prediction[0]}")

Pipeline saved ✓

New Customer Churn Prediction: Yes
